In [41]:
import pandas as pd
import csv

def load_ncu_csv(path):
    # Try multiple separators
    for sep in [';', ',']:
        try:
            df = pd.read_csv(
                path,
                sep=sep,
                engine='python',
                quoting=csv.QUOTE_NONE,
                on_bad_lines='skip'
            )
            if len(df.columns) > 1:
                break
        except:
            continue

    # Clean column names
    df.columns = [c.strip() for c in df.columns]

    print("DEBUG columns:", df.columns.tolist())  # 🔥 inspect once

    # 🔥 Flexible column detection
    metric_col = None
    value_col = None
    kernel_col = None

    for c in df.columns:
        lc = c.lower()
        if "metric" in lc and "name" in lc:
            metric_col = c
        elif "value" in lc or "avg" in lc or "mean" in lc:
            value_col = c
        elif "kernel" in lc:
            kernel_col = c

    if metric_col is None or value_col is None:
        raise ValueError(f"Could not detect required columns. Found: {df.columns.tolist()}")

    # Keep only relevant columns
    cols = [metric_col, value_col]
    if kernel_col:
        cols.append(kernel_col)
        
    print("DEBUG metric columns:", metric_col)  # 🔥 inspect once
    print("DEBUG kernel columns:", kernel_col)  # 🔥 inspect once

    df = df[cols]

    # Rename to standard names
    df = df.rename(columns={
        metric_col: "Metric Name",
        value_col: "Metric Value",
        kernel_col: "Kernel Name" if kernel_col else None
    })

    # Convert values
    df["Metric Value"] = pd.to_numeric(df["Metric Value"], errors="coerce")
    #df = df.dropna(subset=["Metric Value"])

    return df


def pivot_metrics(df):
    if "Kernel Name" in df.columns:
        grouped = df.groupby("Metric Name")["Metric Value"].mean()
    else:
        grouped = df.set_index("Metric Name")["Metric Value"]

    return grouped

In [42]:
def compare_reports(df1, df2, name1="lstsq", name2="normal"):
    merged = pd.concat([df1, df2], axis=1)
    merged.columns = [name1, name2]

    # Convert to numeric where possible
    merged = merged.apply(pd.to_numeric, errors='coerce')

    # Drop NaNs (metrics not shared)
    merged = merged.dropna()

    # Compute differences
    merged["abs_diff"] = merged[name1] - merged[name2]
    merged["pct_diff"] = (merged["abs_diff"] / merged[name2]) * 100

    # Sort by absolute % difference
    merged = merged.sort_values(by="pct_diff", key=abs, ascending=False)

    return merged

In [43]:
df_normal = load_ncu_csv("normalform-50000-128-lstq.csv")
metrics_normal = pivot_metrics(df_normal)

DEBUG columns: ['"ID"', '"Process ID"', '"Process Name"', '"Host Name"', '"Kernel Name"', '"Context"', '"Stream"', '"Block Size"', '"Grid Size"', '"Device"', '"CC"', '"Section Name"', '"Metric Name"', '"Metric Unit"', '"Metric Value"', '"Rule Name"', '"Rule Type"', '"Rule Description"', '"Estimated Speedup Type"', '"Estimated Speedup"']
DEBUG metric columns: "Metric Name"
DEBUG kernel columns: "Kernel Name"


In [44]:
df_normal_100 = load_ncu_csv("normalform-100000-128-lstq.csv")
metrics_normal_100 = pivot_metrics(df_normal_100)

DEBUG columns: ['"ID"', '"Process ID"', '"Process Name"', '"Host Name"', '"Kernel Name"', '"Context"', '"Stream"', '"Block Size"', '"Grid Size"', '"Device"', '"CC"', '"Section Name"', '"Metric Name"', '"Metric Unit"', '"Metric Value"', '"Rule Name"', '"Rule Type"', '"Rule Description"', '"Estimated Speedup Type"', '"Estimated Speedup"']
DEBUG metric columns: "Metric Name"
DEBUG kernel columns: "Kernel Name"


In [45]:
# Pivot
metrics_normal = pivot_metrics(df_normal)
metrics_normal_100 = pivot_metrics(df_normal_100)

In [46]:
# Compare
comparison = compare_reports(metrics_normal, metrics_normal_100)

# Show top differences
print("\n🔥 Top Metric Differences:\n")
print(comparison.head(20))


🔥 Top Metric Differences:

Empty DataFrame
Columns: [lstsq, normal, abs_diff, pct_diff]
Index: []


In [48]:
metrics_normal

Metric Name
 1)"                                                                                                                                                                  NaN
 all instructions are scheduled                                                                                                                                       NaN
 attempt to increase cache hit rates by increasing data locality (coalescing)                                                                                         NaN
 but a per-thread condition code or predicate controls which threads execute the instructions. Try to avoid different execution paths within a warp when possible."   NaN
 reduce the time the active warps are stalled by inspecting the top stall reasons on the Warp State Statistics and Source Counters sections."                         NaN
                                                                                                                                          